# Image classification 

For classification purpose, the output layer needs to be a FC layer with its output the class number. 

ViT needs to append a FC layer (head) because its strucute was originally designed for NLP; for other CNN based model, can modify the last layer in-place.

Fine-tuning can:
1. Update the parameters of the whole model
2. Only update the last layer (or certain layers). Can make these parts require_grad=False.

Notes:
1. By default, pretrained PyTorch models (huggingface) are built considering input data (N,C,H,W) as the first layer or embedding layer. Model itself is built with weights only taking a single data (C,H, W) because no need to duplicated the parameters. For example, the ViT model used in this notebook handles batches in the model embedding layer.
2. PyTorch dataloader will prepare data in correct batch (N,C,H,W). Then you can write outputs=model(batch_data). It will feed each data into the "actual" model, calculate the score per data,and ouput the results in batch.
3. Then you can use criterion to calculate a single loss score per minibatch, and update the weights per minibatch (instead of the whole dataset).
4. PyTorch-Lightning handles batch inside the trainer() class.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import os

# Training
## Parameters

In [2]:
# Models to choose from [resnet, alexnet, vgg, squeezenet, densenet, inception]
model_name = 'ensemble' #"vit_p32" #"resnet" #"squeezenet"

# Number of classes in the dataset
num_classes = 2

# Batch size during training
batch_size = 16

# Number of epochs to train for 
num_epochs = 100

# Flag for feature extracting. When False, we finetune the whole model, when True we only update the reshaped layer params
feature_extract = True

# Number of GPUs available. Use 0 for CPU mode.
ngpu = 1

FT = 2048

# Decide which device we want to run on
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")
print(device)

# output_dir
round_info= 'round2'


cuda:0


## Dataset

In [3]:
transform=transforms.Compose([transforms.Resize(224),
                              transforms.CenterCrop(224),
                              transforms.ToTensor(),
                              transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
                             ])

transform2=transforms.Compose([transforms.Resize(299),
                              transforms.CenterCrop(299),
                              transforms.ToTensor(),
                              transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
                             ])



## Training example

In [4]:
from model_utils import myModels

# Initialize the model for this run
model1_name = "vit_p16"
model2_name = "vit_p32"
model3_name = "resnet"
model4_name = "alexnet"
model5_name = "vgg"
model6_name = "squeezenet"
model7_name = "densenet"
model8_name = "inception"

round2_dir = '/lovelace/xiaoya/scientific_txt2image-main/classifier_checkpoints_3p/'+round_info+'/'

model1, input_size = myModels.initialize_model(model1_name, num_classes, feature_extract, use_pretrained=True)
model1 = torch.load(round2_dir + model1_name + '/' + model1_name + '_' + str(1000) + '_epochs_classifier.pth')

model2, input_size = myModels.initialize_model(model2_name, num_classes, feature_extract, use_pretrained=True)
model2 = torch.load(round2_dir + model2_name + '/' + model2_name + '_' + str(1000) + '_epochs_classifier.pth')

model3, input_size = myModels.initialize_model(model3_name, num_classes, feature_extract, use_pretrained=True)
model3 = torch.load(round2_dir + model3_name + '/' + model3_name + '_' + str(num_epochs) + '_epochs_classifier.pth')

model4, input_size = myModels.initialize_model(model4_name, num_classes, feature_extract, use_pretrained=True)
model4 = torch.load(round2_dir + model4_name + '/' + model4_name + '_' + str(num_epochs) + '_epochs_classifier.pth')

model5, input_size = myModels.initialize_model(model5_name, num_classes, feature_extract, use_pretrained=True)
model5 = torch.load(round2_dir + model5_name + '/' + model5_name + '_' + str(num_epochs) + '_epochs_classifier.pth')

model6, input_size = myModels.initialize_model(model6_name, num_classes, feature_extract, use_pretrained=True)
model6 = torch.load(round2_dir + model6_name + '/' + model6_name + '_' + str(num_epochs) + '_epochs_classifier.pth')

model7, input_size = myModels.initialize_model(model7_name, num_classes, feature_extract, use_pretrained=True)
model7 = torch.load(round2_dir + model7_name + '/' + model7_name + '_' + str(num_epochs) + '_epochs_classifier.pth')

model8, input_size2 = myModels.initialize_model(model8_name, num_classes, feature_extract, use_pretrained=True)
model8 = torch.load(round2_dir + model8_name + '/' + model8_name + '_' + str(num_epochs) + '_epochs_classifier.pth')


Some weights of the model checkpoint at google/vit-base-patch16-224-in21k were not used when initializing ViTModel: ['pooler.dense.bias', 'pooler.dense.weight']
- This IS expected if you are initializing ViTModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing ViTModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of the model checkpoint at google/vit-base-patch32-224-in21k were not used when initializing ViTModel: ['pooler.dense.bias', 'pooler.dense.weight']
- This IS expected if you are initializing ViTModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).


In [5]:
from torchmetrics.image.fid import FrechetInceptionDistance
def predictions_df(dl_model, test_loader, input_size,true_labels):
    pred_hard, pred_soft = [], []
    real_predict,fake_predict=[],[]
    correct_count, all_count = 0,0
    dl_model.eval()
    with torch.no_grad():
        for images, labels in test_loader:
            if torch.cuda.is_available():
                images = images.cuda()
                labels = labels.cuda()
            for i in range(len(labels)):
                img = images[i].view(1, 3, input_size, input_size)
                # soft voting
                output = dl_model(img)
                sm = nn.Softmax(dim=1)
                probabilities = sm(output)
                prob_arr = (probabilities.detach().cpu().numpy())[0]
                # hard voting
                logps = dl_model(img)
                ps = torch.exp(logps)
                probab = list(ps.cpu()[0])
                pred_label = probab.index(max(probab))
                true_label = labels.cpu()[i]
                #exporting to dataframe
                pred_hard.append(pred_label)
                pred_soft.append(prob_arr)
                ####
                if(true_label == pred_label):
                    correct_count += 1
                all_count += 1
                
                if pred_label == 1:
                    real_predict.append(img.detach().cpu().squeeze())
                else: 
                    fake_predict.append(img.detach().cpu().squeeze())
                    
    #list to tensor [N,C,H,W]             
    real_predict = torch.stack(real_predict)  
    fake_predict = torch.stack(fake_predict)  

    #[0,1] to [0,255]
    real_predict = (real_predict.clone().detach()*255).type(torch.uint8)
    fake_predict = (fake_predict.clone().detach()*255).type(torch.uint8)


    print("Number of images predicted as real/fake =", real_predict.shape[0],fake_predict.shape[0], "corrected predictions=", correct_count)
    print("Model Accuracy =",accuracy_score(true_labels,pred_hard))
    print("Model Precision =", precision_score(true_labels,pred_hard))
    print("Model F1 score =", f1_score(true_labels,pred_hard))
    print("Model Recall =", recall_score(true_labels,pred_hard))

    print("\n")
    
    return pred_hard, pred_soft

# Evaluation example (vit_16x16)

In [6]:
import torchvision
from pipeline_utils import Evaluation
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score
from torch.utils.data import ConcatDataset

testroot_rings = "/lovelace/xiaoya/GenAI_dataset/xc_ring_1k_0_100_2nd_corrected"
testroot_peaks = "/lovelace/xiaoya/GenAI_dataset/xc_peaks_5k_0_100_1st"
testroot_empty =  "/lovelace/xiaoya/GenAI_dataset/xc_empty_5k_0_100_1st"

dataset_rings = torchvision.datasets.ImageFolder(root=testroot_rings, transform=transform)
dataset_peaks = torchvision.datasets.ImageFolder(root=testroot_peaks, transform=transform)
dataset_empty = torchvision.datasets.ImageFolder(root=testroot_empty, transform=transform)

dataset = ConcatDataset([dataset_rings,dataset_peaks,dataset_empty])
eval_dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)

true_labels = []
#getting true labels
for images, labels in eval_dataloader:
    for x in range(len(labels)):
        true_labels.append(labels[x].item())

            

dataset_rings2 = torchvision.datasets.ImageFolder(root=testroot_rings, transform=transform2)
dataset_peaks2 = torchvision.datasets.ImageFolder(root=testroot_peaks, transform=transform2)
dataset_empty2 = torchvision.datasets.ImageFolder(root=testroot_empty, transform=transform2)
dataset2 = ConcatDataset([dataset_rings2,dataset_peaks2,dataset_empty2])  

eval_dataloader2 = torch.utils.data.DataLoader(dataset2, batch_size=batch_size, shuffle=False)
true_labels2 = []
#getting true labels
for images, labels in eval_dataloader2:
    for x in range(len(labels)):
        true_labels2.append(labels[x].item())


In [7]:
from sklearn.metrics import accuracy_score
from numpy import cov
# from keras.applications.inception_v3 import InceptionV3
# model =  InceptionV3(include_top=False, pooling='avg', input_shape=(224,224,3))
# model2 =  InceptionV3(include_top=False, pooling='avg', input_shape=(299,299,3))

model1_hard, model1_soft = predictions_df(model1, eval_dataloader, input_size,true_labels)
model2_hard, model2_soft = predictions_df(model2, eval_dataloader, input_size,true_labels)
model3_hard, model3_soft = predictions_df(model3, eval_dataloader, input_size,true_labels)
model4_hard, model4_soft = predictions_df(model4, eval_dataloader, input_size,true_labels)
model5_hard, model5_soft = predictions_df(model5, eval_dataloader, input_size,true_labels)
model6_hard, model6_soft = predictions_df(model6, eval_dataloader, input_size,true_labels)
model7_hard, model7_soft = predictions_df(model7, eval_dataloader, input_size,true_labels)
model8_hard, model8_soft = predictions_df(model8, eval_dataloader2, input_size2,true_labels2)

Number of images predicted as real/fake = 293 307 corrected predictions= 533
Model Accuracy = 0.8883333333333333
Model Precision = 0.8976109215017065
Model F1 score = 0.8870151770657673
Model Recall = 0.8766666666666667


Number of images predicted as real/fake = 296 304 corrected predictions= 532
Model Accuracy = 0.8866666666666667
Model Precision = 0.8918918918918919
Model F1 score = 0.8859060402684564
Model Recall = 0.88


Number of images predicted as real/fake = 290 310 corrected predictions= 498
Model Accuracy = 0.83
Model Precision = 0.8413793103448276
Model F1 score = 0.8271186440677966
Model Recall = 0.8133333333333334


Number of images predicted as real/fake = 292 308 corrected predictions= 496
Model Accuracy = 0.8266666666666667
Model Precision = 0.8356164383561644
Model F1 score = 0.8243243243243243
Model Recall = 0.8133333333333334


Number of images predicted as real/fake = 304 296 corrected predictions= 514
Model Accuracy = 0.8566666666666667
Model Precision = 0.8519736

In [8]:
#!pip install pandas


df_hard_voting = pd.DataFrame.from_dict({'vit16':model1_hard, 'vit32':model2_hard,
                    'resnet':model3_hard, 'alexnet':model4_hard, 'vgg':model5_hard, 'squeezenet':model6_hard, 'densenet':model7_hard, 'inception':model8_hard})


df_soft_voting = pd.DataFrame.from_dict({'vit16_soft':model1_soft, 'vit32_soft':model2_soft,
                    'resnet_soft':model3_soft, 'alexnet_soft':model4_soft, 'vgg_soft':model5_soft, 'squeezenet_soft':model6_soft, 'densenet_soft':model7_soft, 'inception_soft':model8_soft})

In [9]:
ensemble_hard_predictions = np.asarray(df_hard_voting.mode(axis=1)[0])
ensemble_hard_score = accuracy_score(np.asarray(true_labels), ensemble_hard_predictions)
ensemble_hard_precision = precision_score(np.asarray(true_labels), ensemble_hard_predictions)
ensemble_hard_f1 = f1_score(np.asarray(true_labels), ensemble_hard_predictions)
ensemble_hard_recall = recall_score(np.asarray(true_labels), ensemble_hard_predictions)
print(f"The Accuracy Score and precision of Hard Voting Ensemble is:  {(ensemble_hard_score*100):.4f} %, {(ensemble_hard_precision*100):.4f} %")
print(f"The F1 Score and recall of Hard Voting Ensemble is:  {(ensemble_hard_f1*100):.4f} %, {(ensemble_hard_recall*100):.4f} %")

The Accuracy Score and precision of Hard Voting Ensemble is:  90.3333 %, 90.8784 %
The F1 Score and recall of Hard Voting Ensemble is:  90.2685 %, 89.6667 %


In [10]:
import operator
def get_soft_voting():
    preds = []
    for x in range(len(df_soft_voting)):
        sample = (0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
        for y in range(len(df_soft_voting.columns)):
            sample = tuple(map(operator.add, sample, (tuple(df_soft_voting.iloc[x,y]))))
        sample = tuple(ti/len(sample) for ti in sample)
        element = max(sample)
        idx = sample.index(element)
        preds.append(idx)
    return preds

ensemble_soft_preds = get_soft_voting()
ensemble_soft_score = accuracy_score(np.asarray(true_labels), np.asarray(ensemble_soft_preds))
ensemble_soft_precision = precision_score(np.asarray(true_labels), np.asarray(ensemble_soft_preds))
ensemble_soft_f1 = f1_score(np.asarray(true_labels), np.asarray(ensemble_soft_preds))
ensemble_soft_recall = recall_score(np.asarray(true_labels), np.asarray(ensemble_soft_preds))
print(f"The Accuracy Score and precision of Soft Voting (unweighted) Ensemble is:  {(ensemble_soft_score*100):.4f} %,{(ensemble_soft_precision*100):.4f} %")
print(f"The F1 Score and recall of Soft Voting (unweighted) Ensemble is:  {(ensemble_soft_f1*100):.4f} %,{(ensemble_soft_recall*100):.4f} %")

The Accuracy Score and precision of Soft Voting (unweighted) Ensemble is:  90.0000 %,89.2157 %
The F1 Score and recall of Soft Voting (unweighted) Ensemble is:  90.0990 %,91.0000 %


In [13]:
import operator
def get_weighted_average():
    preds = []
    #weights =[0.4,0.6,0.3,0.1,0.3,0.3,0.4,0.2]
    #weights =[0.7,0.8,0.3,0.1,0.3,0.3,0.4,0.2]
    weights =[0.9,0.8,0.3,0.1,0.3,0.3,0.4,0.2]

    for x in range(len(df_soft_voting)):
        sample = (0.0, 0.0)
        for y in range(len(df_soft_voting.columns)):
            ##
            k = tuple(float(weights[y]) * element for element in (tuple(df_soft_voting.iloc[x,y])))
            ##
            sample = tuple(map(operator.add, sample, k))
        sample = tuple(ti/len(sample) for ti in sample)
        element = max(sample)
        idx = sample.index(element)
        preds.append(idx)
    return preds

weighted_soft_preds = get_weighted_average()
weighted_soft_score = accuracy_score(np.asarray(true_labels), np.asarray(weighted_soft_preds))
weighted_soft_precision = precision_score(np.asarray(true_labels), np.asarray(weighted_soft_preds))
weighted_soft_f1 = f1_score(np.asarray(true_labels), np.asarray(weighted_soft_preds))
weighted_soft_recall = recall_score(np.asarray(true_labels), np.asarray(weighted_soft_preds))
print(f"The Accuracy Score and Precision of Soft Voting (weighted) Ensemble is:  {(weighted_soft_score*100):.4f} %,   {(weighted_soft_precision*100):.4f} %")
print(f"The F1 Score and recall of Soft Voting (weighted) Ensemble is:  {(weighted_soft_f1*100):.4f} %,   {(weighted_soft_recall*100):.4f} %")

The Accuracy Score and Precision of Soft Voting (weighted) Ensemble is:  90.6667 %,   90.1316 %
The F1 Score and recall of Soft Voting (weighted) Ensemble is:  90.7285 %,   91.3333 %


In [12]:
# from PIL import Image

# dataset = torchvision.datasets.ImageFolder(root="/lovelace/xiaoya/GenAI_dataset/xc_ring_1k_0_100_2nd_corrected", transform=transform)
# eval_dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)

# real_test_images=[]
# fake_test_images=[]
# all_test_images=[]
# exp_images=[]
# for images, labels in eval_dataloader:
#     for x in range(len(labels)):
#         all_test_images.append(images[x])

# ensemble_real=[]
# ensemble_fake=[]
# for i in range(len(weighted_soft_preds)):
#     if weighted_soft_preds[i]==1:
#         ensemble_real.append(all_test_images[i])
#     else:
#         ensemble_fake.append(all_test_images[i])

# for i in range(len(ensemble_real)):
#     savepath="/lovelace/xiaoya/GenAI_dataset/ensemble_prediction/"
#     filename = str(i)+".jpg"
#     arr = ensemble_real[i].permute(1,2,0).numpy()*255
#     im = Image.fromarray(arr.astype(np.uint8))
#     im.save(savepath+filename)
    